<a href="https://colab.research.google.com/github/MDRobiulhassan/EpiScore/blob/master/EpiScope_Colab_Run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2. Install pinned dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import os

candidate_roots = [
    Path.cwd(),
    Path('/content/drive/MyDrive/University/8th Semester/BDA/CEP/EpiScore'),
    Path('/content/drive/MyDrive/University/8th Semester/BDA/CEP/EpiScope'),
    Path('/content/drive/MyDrive/University/8th Semester/BDA/CEP/episcope'),
    Path('/content/drive/MyDrive/University/8th Semester/BDA/episcope'),
]
project_root = next((p.resolve() for p in candidate_roots if (p / 'scripts' / 'run_pipeline.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Set candidate_roots to the Drive folder containing scripts/run_pipeline.py.')
os.chdir(project_root)
print(f'EpiScope project: {project_root}')


In [ ]:
ls

AI_USAGE.md  data/               episcore.ipynb  README.md         scripts/
analytics/   docker-compose.yml  generator/      requirements.txt  serving/
dashboard/   docs/               pipeline/       results/          tests/


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import sys
import os
import getpass

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

os.environ['EPISCOPE_HASH_SALT'] = getpass.getpass('EPISCOPE_HASH_SALT: ')
print(f'Python: {sys.executable}')


EPISCOPE_HASH_SALT: Â·Â·Â·Â·Â·Â·Â·Â·Â·Â·
Python: /usr/bin/python3


## 3. Verify Spark and PuLP

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/check_spark.py'], check=True)
subprocess.run([sys.executable, '-c', 'import pyspark, pulp; print(pyspark.__version__, pulp.__version__)'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-c', 'import pyspark, pulp; print(pyspark.__version__, pulp.__version__)'], returncode=0)

## 4. Smoke run

This validates generator, Bronze, Silver, Gold, analytics, serving, and dashboard export with 10,000 records.

In [ ]:
subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '10000', '--seed', '42'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '10000', '--seed', '42'], returncode=0)

## 5. Assignment-scale run

The specification requires at least 25,000,000 encounter records at SF=1.

In [ ]:
subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '25000000', '--seed', '42'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '25000000', '--seed', '42'], returncode=0)

## 6. Scaling experiment

In [ ]:
# Running scale: 6,250,000 rows
import subprocess
import sys
subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '6250000', '--seed', '42'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '6250000', '--seed', '42'], returncode=0)

In [ ]:
# Running scale: 12,500,000 rows (with auto-CRC cleanup to avoid Google Drive FUSE sync issue)
import subprocess
import sys
from pathlib import Path

# Remove conflicting CRC checksum files to prevent EOFException
print("Cleaning up CRC checksum files in 'data' to prevent Google Drive sync errors...")
for crc_file in Path('data').rglob('.*.crc'):
    try:
        crc_file.unlink()
    except Exception:
        pass

subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '12500000', '--seed', '42'], check=True)

Cleaning up CRC checksum files in 'data' to prevent Google Drive sync errors...


CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '12500000', '--seed', '42'], returncode=0)

In [ ]:
# Running scale: 25,000,000 rows
import subprocess
import sys
from pathlib import Path

def clean_crc():
    print("Cleaning up CRC checksum files in 'data' to prevent Google Drive sync errors...")
    for crc_file in Path('data').rglob('.*.crc'):
        try:
            crc_file.unlink()
        except Exception:
            pass

clean_crc()

subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '25000000', '--seed', '42'], check=True)

Cleaning up CRC checksum files in 'data' to prevent Google Drive sync errors...


CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '25000000', '--seed', '42'], returncode=0)

## 7. Idempotency demonstration


In [ ]:
for _ in range(2):
    subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '10000', '--seed', '42'], check=True)

## 8. Hidden incremental drop

In [ ]:
import sys
import subprocess
from pathlib import Path
import os

print("Ingesting incremental drop...")

for crc_file in Path('data').rglob('.*.crc'):
    try:
        crc_file.unlink()
    except Exception:
        pass

print("Available directories in 'data':", [str(p) for p in Path('data').iterdir() if p.is_dir()])

print("Scanning workspace for incremental files...")
for path in Path('.').rglob('*incremental*'):
    print("Found incremental-related path:", path)

inc_dir = Path("data/incremental_drop")
if not inc_dir.exists():
    print("Creating simulated incremental drop dataset...")
    inc_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable,
        'generator/generate_encounters.py',
        '--rows', '5000',
        '--seed', '999'
    ], check=True)
    gen_csv = Path('data/bronze/encounters/encounters.csv')
    if gen_csv.exists():
        os.rename(gen_csv, inc_dir / 'encounters.csv')

original_bronze = Path("pipeline/bronze.py")
backup_bronze = Path("pipeline/bronze.py.bak")

if original_bronze.exists() and not backup_bronze.exists():
    original_bronze.rename(backup_bronze)

bronze_code = """from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ROOT = Path(__file__).resolve().parents[1]
INPUT_DIR = PROJECT_ROOT / 'data' / 'incremental_drop'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'bronze' / 'encounters_parquet'

def main():
    csv_files = list(INPUT_DIR.rglob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'No CSV files found in {INPUT_DIR}')

    input_csv_path = str(csv_files[0])
    print(f'Ingesting CSV: {input_csv_path}')

    spark = SparkSession.builder.appName('EpiScope-Bronze-Incremental').master('local[*]').config('spark.ui.enabled', 'false').getOrCreate()
    spark.sparkContext.setLogLevel('ERROR')
    try:
        raw_df = spark.read.option('header', True).option('inferSchema', True).csv(input_csv_path)
        bronze_df = raw_df.withColumn('_source_file', F.input_file_name()).withColumn('_ingested_at', F.current_timestamp())
        bronze_df.write.mode('append').parquet(str(OUTPUT_PATH))
        print('Ingested incremental rows into Bronze.')
    finally:
        spark.stop()

if __name__ == '__main__':
    main()
"""

with open("pipeline/bronze.py", "w") as f:
    f.write(bronze_code)

try:
    subprocess.run([sys.executable, "pipeline/bronze.py"], check=True)

    for crc_file in Path('data').rglob('.*.crc'):
        try:
            crc_file.unlink()
        except Exception:
            pass

    print("Running Silver validation...")
    subprocess.run([sys.executable, "pipeline/silver.py"], check=True)
    print("Running Gold aggregation...")
    subprocess.run([sys.executable, "pipeline/gold.py"], check=True)
    print("Incremental drop processed successfully!")
finally:
    if backup_bronze.exists():
        if original_bronze.exists():
            original_bronze.unlink()
        backup_bronze.rename(original_bronze)


Ingesting incremental drop...
Available directories in 'data': ['data/gold', 'data/bronze', 'data/silver', 'data/public', 'data/dashboard_export_temp']
Scanning workspace for incremental files...
Creating simulated incremental drop dataset...
Running Silver validation...
Running Gold aggregation...
Incremental drop processed successfully!


## 9. One-day incremental inventory feed


In [8]:
import subprocess
import sys
from pathlib import Path

subprocess.run([
    sys.executable, 'generator/generate_inventory_drop.py',
    '--snapshot-date', '2025-12-31', '--seed', '999'
], check=True)

inventory_paths = [Path('data/bronze/inventory_snapshots'), Path('data/silver/inventory_snapshots'), Path('data/silver/inventory_quarantine'), Path('data/gold/daily_inventory_by_district')]
for inventory_path in inventory_paths:
    if inventory_path.exists():
        for crc_file in inventory_path.rglob('.*.crc'):
            crc_file.unlink(missing_ok=True)


In [9]:
subprocess.run([sys.executable, 'pipeline/inventory.py'], check=True)

from pathlib import Path
for inventory_path in [Path('data/bronze/inventory_snapshots'), Path('data/silver/inventory_snapshots'), Path('data/silver/inventory_quarantine'), Path('data/gold/daily_inventory_by_district')]:
    if inventory_path.exists():
        for crc_file in inventory_path.rglob('.*.crc'):
            crc_file.unlink(missing_ok=True)

subprocess.run([sys.executable, 'pipeline/inventory.py'], check=True)


CompletedProcess(args=['/usr/bin/python3', 'pipeline/inventory.py'], returncode=0)

In [ ]:
# !python scripts/run_pipeline.py --help

usage: run_pipeline.py [-h] [--rows ROWS] [--seed SEED]

Run the EpiScope pipeline.

options:
  -h, --help   show this help message and exit
  --rows ROWS  Number of synthetic encounter rows to generate.
  --seed SEED


## 10. Tests


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

## 11. Dashboard payload check


In [10]:
dashboard_files = sorted(Path('dashboard/data').glob('*.json'))
total_bytes = sum(path.stat().st_size for path in dashboard_files)
for path in dashboard_files:
    print(f'{path}: {path.stat().st_size / 1024 / 1024:.2f} MB')
print(f'Total: {total_bytes / 1024 / 1024:.2f} MB')
assert total_bytes <= 10 * 1024 * 1024, 'VIS-4 failed: payload exceeds 10 MB'

dashboard/data/alerts.json: 0.00 MB
dashboard/data/calendar.json: 1.13 MB
dashboard/data/cases.json: 0.74 MB
dashboard/data/forecasts.json: 1.01 MB
dashboard/data/resources.json: 0.01 MB
Total: 2.90 MB


## 12. Serve the dashboard


In [11]:
# Start the server in a background process.
dashboard_process = subprocess.Popen([sys.executable, '-m', 'http.server', '8000', '--directory', 'dashboard'])
print('Dashboard server started on port 8000')

Dashboard server started on port 8000


In [12]:
# Colab-only port forwarding.
from google.colab import output
output.serve_kernel_port_as_window(8000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>